In [ ]:
from astropy.io import fits
from astropy.coordinates import SkyCoord
from astropy.coordinates import ICRS, Galactic, FK4, FK5
import numpy as np
import matplotlib.pyplot as plt
from reproject.mosaicking import find_optimal_celestial_wcs
from reproject import reproject_interp
from reproject.mosaicking import reproject_and_coadd
from astropy.wcs import WCS
from astropy.utils.data import get_pkg_data_filename
from astropy.convolution import Gaussian2DKernel
#from scipy.signal import convolve as scipy_convolve
from astropy.convolution import convolve
import gc
from matplotlib.patches import Circle
from scipy.stats import linregress
import matplotlib as mpl
from matplotlib.gridspec import GridSpec

In [ ]:
hdu_w06q = fits.open('/srv/data/cgps/drao26m_2006/q_gal.fit')
hdu_w06u = fits.open('/srv/data/cgps/drao26m_2006/u_gal.fit')
hdu_hbn  = fits.open('/srv/data/gmims/gmims-hbn/GMIMS-HBN_v1_gal_car_freq_IQU.fits')

wcs_w06 = WCS(hdu_w06q[0].header)
wcs_hbn = WCS(hdu_hbn[0].header).dropaxis(3).dropaxis(2)

print(wcs_w06)
print(wcs_hbn)

PI_w06 = np.sqrt(hdu_w06q[0].data**2 + hdu_w06u[0].data**2)/1e3
PI_hbn = np.sqrt(hdu_hbn[0].data[1]**2 + hdu_hbn[0].data[2]**2)


In [ ]:
fig,ax = plt.subplots(1,2,figsize=(12,12))
ax[0].imshow(PI_hbn[121],origin='lower',vmin=0,vmax=0.5)
ax[1].imshow(PI_w06,origin='lower',vmin=0,vmax=0.5)

In [ ]:
fig = plt.figure(figsize=(20,16))
gs = GridSpec(8, 1, figure=fig, height_ratios=[1,1,0.2,1,1,0.2,1,1])

fs = 12

ax1 = fig.add_subplot(gs[0], projection=wcs_w06_fix)
ax2 = fig.add_subplot(gs[1], projection=wcs_hbn_fix)
ax3 = fig.add_subplot(gs[3], projection=wcs_w06_fix)
ax4 = fig.add_subplot(gs[4], projection=wcs_hbn_fix)
ax5 = fig.add_subplot(gs[6], projection=wcs_w06_fix)
ax6 = fig.add_subplot(gs[7], projection=wcs_hbn_fix)


cx = SkyCoord([100, 50], [0, 0], frame=Galactic, unit="deg")
cy = SkyCoord([75,  75], [-6,8], frame=Galactic, unit="deg")

im1 = ax1.imshow(PI_w06_fix, origin='lower', vmin=0, vmax=0.6, cmap='gist_heat_r')
ax1.set_xlim(wcs_w06_fix.world_to_pixel(cx)[0])
ax1.set_ylim(wcs_w06_fix.world_to_pixel(cy)[1])

im2 = ax2.imshow(PI_hbn_fix, origin='lower', vmin=0, vmax=0.6, cmap='gist_heat_r')
ax2.set_xlim(wcs_hbn_fix.world_to_pixel(cx)[0])
ax2.set_ylim(wcs_hbn_fix.world_to_pixel(cy)[1])


cx = SkyCoord([150, 100], [0, 0], frame=Galactic, unit="deg")
cy = SkyCoord([125, 125], [-6,8], frame=Galactic, unit="deg")

im3 = ax3.imshow(PI_w06_fix, origin='lower', vmin=0, vmax=0.6, cmap='gist_heat_r')
ax3.set_xlim(wcs_w06_fix.world_to_pixel(cx)[0])
ax3.set_ylim(wcs_w06_fix.world_to_pixel(cy)[1])

im4 = ax4.imshow(PI_hbn_fix, origin='lower', vmin=0, vmax=0.6, cmap='gist_heat_r')
ax4.set_xlim(wcs_hbn_fix.world_to_pixel(cx)[0])
ax4.set_ylim(wcs_hbn_fix.world_to_pixel(cy)[1])


cx = SkyCoord([200, 150], [0, 0], frame=Galactic, unit="deg")
cy = SkyCoord([175, 175], [-6,8], frame=Galactic, unit="deg")

im5 = ax5.imshow(PI_w06_fix, origin='lower', vmin=0, vmax=0.6, cmap='gist_heat_r')
ax5.set_xlim(wcs_w06_fix.world_to_pixel(cx)[0])
ax5.set_ylim(wcs_w06_fix.world_to_pixel(cy)[1])

im6 = ax6.imshow(PI_hbn_fix, origin='lower', vmin=0, vmax=0.6, cmap='gist_heat_r')
ax6.set_xlim(wcs_hbn_fix.world_to_pixel(cx)[0])
ax6.set_ylim(wcs_hbn_fix.world_to_pixel(cy)[1])

In [ ]:
print((50+100)/2)
print((100+150)/2)
print((150+200)/2)

In [ ]:
PI_w06_fix = np.empty_like(PI_w06)
print(PI_w06_fix.shape)
PI_w06_fix[:,0:720] = PI_w06[:,720:1440]
PI_w06_fix[:,720:1440] = PI_w06[:,0:720]

PI_hbn_fix = np.empty_like(PI_hbn[121])
print(PI_hbn_fix.shape)
PI_hbn_fix[:,0:720] = PI_hbn[121,:,720:1440]
PI_hbn_fix[:,720:1440] = PI_hbn[121,:,0:720]

hdr_w06_fix = wcs_w06.copy().to_header()
print(repr(hdr_w06_fix))
hdr_w06_fix['CRVAL1'] = 180.0
wcs_w06_fix = WCS(hdr_w06_fix)
print('')
hdr_hbn_fix = wcs_hbn.copy().to_header()
print(repr(hdr_hbn_fix))
hdr_hbn_fix['CRVAL1'] = 180.0
wcs_hbn_fix = WCS(hdr_hbn_fix)
#hdr_w06_fix['CRPIX2'] = 1
#hdr_w06_fix['C']
#hdr_w06_fix['']
#print(hdr_w06_fix)

In [ ]:
fig = plt.figure(figsize=(16,12))
gs = GridSpec(2, 1, figure=fig)

fs = 12

ax1 = fig.add_subplot(gs[0], projection=wcs_w06_fix)
ax2 = fig.add_subplot(gs[1], projection=wcs_hbn_fix)

ax1.imshow(PI_w06_fix, origin='lower', vmin=0, vmax=0.6, cmap='gist_heat_r')
ax2.imshow(PI_hbn_fix, origin='lower', vmin=0, vmax=0.6, cmap='gist_heat_r')

In [ ]:
c = SkyCoord(llim, blim, frame=Galactic, unit="deg")
fs = 22
    
fig = plt.figure(figsize=(20,12))
    
plt.subplots_adjust(hspace=0.0,left=0.08, right=0.98, top=0.99, bottom=0.08)
    
    cmap = mpl.colormaps.get_cmap(cmap1)  # viridis is the default colormap for imshow
    cmap.set_bad(color='grey')
    
    ax1  = fig.add_subplot(211, projection=WCS(hdr).celestial)
    im1  = ax1.imshow(data1, origin='lower', vmin=-v1max, vmax=v1max,cmap=cmap)
    ax1.set_xlim(WCS(hdr).world_to_pixel(c)[0])
    ax1.set_ylim(WCS(hdr).world_to_pixel(c)[1])
    #ax1.set_xticks([125,130,135])
    cbar1 = fig.colorbar(im1, ax=ax1, orientation='vertical',fraction=0.1,pad=0.0,aspect=15)
    cbar1.set_label(r'RM (rad m$^{-2}$)', fontsize=fs)
    cbar1.set_ticks([-200,-100,0,100,200])

    cmap = mpl.colormaps.get_cmap(cmap2)  # viridis is the default colormap for imshow
    cmap.set_bad(color='grey')
   
    ax2  = fig.add_subplot(212, projection=WCS(hdr).celestial)
    im2  = ax2.imshow(data2, origin='lower', vmin=0, vmax=v2max,cmap=cmap)
    ax2.set_xlim(WCS(hdr).world_to_pixel(c)[0])
    ax2.set_ylim(WCS(hdr).world_to_pixel(c)[1])
    #ax2.set_xticks([125,130,135])
    cbar2 = fig.colorbar(im2, ax=ax2, orientation='vertical',fraction=0.1,pad=0.0,aspect=15)
    cbar2.set_label(r'PI (K)', fontsize=fs)
    cbar2.set_ticks([0,0.1,0.2,0.3,0.4,0.5])
    
    
    ax2.set_xlabel('Galactic Longitude',fontsize=fs)
    fig.text(0.02,0.45,'Galactic Latitude',fontsize=fs,rotation='vertical')
    for ax in [ax1,ax2]:
        ax.tick_params(axis='both', labelsize=fs)
        ax.set_ylabel('  ',fontsize=fs)
        ax.tick_params(axis='both', which='both', width=2, length=6)
        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_linewidth(2)
        
    for cbar in [cbar1,cbar2]:
        cbar.ax.tick_params(axis='y', which='both', width=2, length=6)
        cbar.ax.tick_params(labelsize=fs)
        cbar.outline.set_linewidth(2)
    
    #plt.savefig('/home/aordog/CGPS_GMIMS_PLOTS/'+filename+'.pdf')
    plt.savefig('../plots/review_tests/'+filename+'.pdf')
